# SDF+

Determining signed distance to curve and $t$ of closest point.


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
import math
import numpy as np

arr = np.array

In [ ]:
import pyvista as pv
from matplotlib.cm import tab10

tab10colors = tab10.colors  # type: ignore
yrnclr = [tab10.colors[4], tab10.colors[6]]

In [ ]:
from splines import M1, M2, mega_curve, segm_curve, curve_t


def spline1(points):
    return mega_curve(1, M1, points, arr((0.0, 1.0)))

In [ ]:
RES = 64
PIX = 1.0 / RES
X, Y = np.meshgrid(np.arange(RES - 1), np.arange(RES - 1))
PP = (np.column_stack([X.flatten(), Y.flatten()]) + 0.5) * PIX

In [ ]:
N = 6
ls = np.linspace(0.0, 1.0, N)
rndr = 2.0 / N
points = arr([
    ls + rndr * (np.random.random(N) - 0.5),
    ls + rndr * (np.random.random(N) - 0.5),
    np.zeros(N) + 0.1,
]).T

points2d = points[:, :2]

In [ ]:
pl = pv.Plotter()
pl.background_color = pv.Color("#303030")
pl.add_mesh(pv.lines_from_points(points), color="black", line_width=2)
pl.view_xy()
pl.show()

In [ ]:
grid = pv.ImageData(dimensions=(RES, RES, 1), spacing=(PIX, PIX, PIX))
_ = pl.add_mesh(grid, name="tex", cmap="seismic", clim=[-1.0, +1.0], interpolate_before_map=False)

In [ ]:
def on_click(point):
    pl.add_text(f"clicked: {point}", name="clicked", position="lower_left")


pl.enable_point_picking(callback=on_click)

# Orientation

- $T(t')$ — direction of spline in UV plane, should be consistent all along the spline
- $P = p_s - S(t)$ ­— vector from curve to a sample point

Facts:

- $|P · T| = |P||T|cos \phi$
- $|P × T| = |P||T|sin \phi$

$$
(P × T)_z =
\begin{cases}
> 0 \implies 0 < \phi < 180° \implies P ~ \text{is on right-hand side of the curve} \\
= 0 \implies \phi = 0 \implies P ~ \text{is straight forward (an edge case)} \\
< 0 \implies -180° < \phi < 0 \implies P ~ \text{is on left-hand side of the curve} \\
\end{cases}
$$


## Points spline

Just points, distance to them


In [ ]:
def disq0(p1, ps):
    s = ps - p1
    return s @ s


def dist0map(ps):
    d_ = [disq0(points2d[i], ps) for i in range(N)]
    d = min(d_)
    return math.sqrt(d)

In [ ]:
values = np.fromiter(map(dist0map, PP), dtype=float)

In [ ]:
grid.cell_data["scalars"] = values


## Linear spline

For a local segment $\big[ P_1, P_2 \big]$ and sample point $P_s$ (localized to $P_0$)

- $P'_1 = P_1 - P_1 = 0$
- $P'_2 = P_2 - P_1$
- $P'_s = P_s - P_1$
- $l = |P_2 - P_1|^2 = P'_2 \cdot P'_2$ — segment squared length
- $j = P'_s \cdot P'_2$ — projlength of sample to segment
- $t = j / l $ — relative proj length, could be outside $[0, 1]$
- $r = t P'_2$ — vect to closest point on the segment (from P_1)
    - clamped(t) -> distance to edge points
- $o = P'_s - r$ — perpendicular from sample to line
- $d = |o|^2 = o \cdot o$ — squared distance from the sample to line


In [ ]:
def dist1_t(p1, p2, ps):
    ps_ = ps - p1
    p2_ = p2 - p1
    l = p2_ @ p2_
    t = (ps_ @ p2_) / l
    return min(1.0, max(0.0, t))


def disq1(p1, p2, t, ps):
    ps_ = ps - p1
    p2_ = p2 - p1
    o = ps_ - t * p2_
    return o @ o


def dist1map(p):
    t_ = [dist1_t(points2d[i], points2d[i + 1], p) for i in range(N - 1)]
    d_ = [disq1(points2d[i], points2d[i + 1], t_[i], p) for i in range(N - 1)]
    d_min = min(d_)
    return math.sqrt(d_min)

In [ ]:
values = np.fromiter(map(dist1map, PP), dtype=float)

In [ ]:
grid.cell_data["scalars"] = values.flatten("F")
_ = pl.add_mesh(grid, name="tex", cmap="seismic", clim=[-1.0, +1.0], interpolate_before_map=False)

# Quadratic

For a segment $S_i(t)$ with control points $c_1 = c_{i-2}, c_2 = c_{i-1}, c_3 = c_{i}$ and local $t \in [0, 1)$

$$
S(t) =
(1/2)
\big[1, t, t^2 \big]
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix}
\begin{bmatrix}
c_1 \\
c_2 \\
c_3 \\
\end{bmatrix}
$$

$\big[1, t, t^2 \big]
\begin{bmatrix}
½ (c_1 + c_2) \\
c_2 - c_1 \\
½ (c_1 + c_3) - c_2 
\end{bmatrix}$

With midpoints and symmetry around $c_2:

- $m_1 = ½(c_1 + c_2)$
- $m_2 = ½(c_1 + c_3)$
- $m_3 = ½(c_3 + c_2)$
- $v_1 = m_1 - c_2 = ½(c_1 - c_2)$
- $v_3 = m_3 - c_2 = ½(c_3 - c_2)$
- $v_2 = m_2 - c_2 = v_1 + v_3 $ for no reason

$S(t) = c_2 + \big[1, t, t^2 \big]
\begin{bmatrix}
v_1 \\
-2 v_1 \\
v_2
\end{bmatrix}$


In [ ]:
spline2 = mega_curve(2, M2, points, np.linspace(0.0, 1.0, 8))

_ = pl.add_mesh(pv.lines_from_points(spline2), name="spline", line_width=2, color="gray")

### Equation

Of the closest point

- $p$ — sample point
- $S(t)$ — some point on curve
- $(p - S(t))$ — vector from curve to the sample
- $S'(t)$ — derivative = tangent of the curve at that point

$$(p - S(t)) \cdot S'(t) = 0$$

The solutions determine points where vector from curve to the sample perpendicular to curve tangent.


- $S = \big[1,t,t^2\big] \odot \big[c_2 + v_1, -2v_1, v_2\big]$
- $S' = \big[1,t\big] \odot \big[-2v_1, 2v_2\big]$
- $p_s = p - c_2$
- $p_s - S = \big[1,t,t^2\big] \odot \big[p_s - v_1, 2v_1, -v_2\big]$

* `00`: $(1) -2 (p_s·v_1 - v_1^2)$
* `01`: $(t) 2 (p_s·v_2 - v_1·v_2)$
* `10`: $(t) -4 (v_1^2)$
* `11`: $(t^2) 4 (v_1·v_2)$
* `20`: $(t^2) 2 (v_1·v_2)$
* `21`: $(t^3) -2 (v_2^2)$

$(p_s - S) · S' = \big[1,t,t^2,t^3\big] \odot \big[-2 p_s·v_1, 2 p_s·v_2, 0, 0\big] + 2 \big[v_1·v_1, -v_1·v_2 -2 v_1·v_1, 3 v_1·v_2, -v_2·v_2\big]$


In [ ]:
pointsC1 = points2d[:-2, :]
pointsC2 = points2d[1:-1, :]
pointsC3 = points2d[2:, :]

vectorsV1 = (pointsC1 - pointsC2) * 0.5
vectorsV3 = (pointsC3 - pointsC2) * 0.5
vectorsV2 = vectorsV1 + vectorsV3

dotsV11 = arr([v1 @ v1 for v1 in vectorsV1])
dotsV22 = arr([v2 @ v2 for v2 in vectorsV2])
dotsV12 = arr([v1 @ v2 for v1, v2 in zip(vectorsV1, vectorsV2)])

params = arr([dotsV11, -1.0 * dotsV12 - 2.0 * dotsV11, 3.0 * dotsV12, -1.0 * dotsV22]).T * 2

In [ ]:
def paramsp(i, p):
    p_s = p - pointsC2[i]
    return params[i] + arr([-2 * (p_s @ vectorsV1[i]), 2 * (p_s @ vectorsV2[i]), 0, 0])

## Cubic equation

$$a x^3 + b x^2 + c x + d = 0$$

$$t^3 + p t + q = 0$$

Solvable in closed form...


In [ ]:
def dist2_t(i, p):
    a_0, a_1, a_2, a_3 = paramsp(i, p)
    roots = [r for r in np.roots([a_3, a_2, a_1, a_0]) if r.imag == 0.0]
    if len(roots) == 0:
        return None
    if len(roots) == 3:
        t = roots[1].real
    else:
        t = roots[0].real
    return t


def dist2_m(i, p):
    a_0, a_1, a_2, a_3 = paramsp(i, p)
    roots = [r.real for r in np.roots([a_3, a_2, a_1, a_0]) if r.imag == 0.0]
    print(roots)
    return len(roots)


def disq2(points, t, p):
    if t is None:
        return math.inf
    t = min(1.0, max(0.0, t))
    pj = curve_t(2, M2, points, t)
    o = p - pj
    return o @ o


def dist2map(p):
    t_ = [dist2_t(i, p) for i in range(N - 2)]
    d_ = [disq2(points2d[i : i + 3], t_[i], p) for i in range(N - 2)]
    d_min = min(d_)
    return math.sqrt(d_min)

In [ ]:
values = np.fromiter(map(dist2map, PP), dtype=float)
# values = np.fromiter(map(lambda p: dist2_m(0, p), PP), dtype=float)

In [ ]:
grid.cell_data["scalars"] = values
_ = pl.add_mesh(grid, name="tex", cmap="seismic", clim=[-1.0, +1.0], interpolate_before_map=False)
# _ = pl.add_mesh(grid, name="tex", cmap="tab10")